# Biological Grounding

Live mode recomputes Frank PST synthetic conditions and uses the real local extracted dopamine cache when available. EEG/OpenNeuro panels are explicitly external-data gated.

### Setup and Dependencies
Imports the trace package, plotting utilities, and configures the default execution mode.


In [ ]:
import os, math, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

RESULT_MODE = "live"
if RESULT_MODE not in {"live", "full_sweep_cache"}:
    raise ValueError("RESULT_MODE must be 'live' or 'full_sweep_cache'")
GREEN, INDIGO, RED, GOLD, GREY, PURPLE, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#b07cc6", "#2b2b2b"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

def _cache(name):
    print(f"FULL-SWEEP CACHE: {paths.results_dir() / name}")
    return paths.load_result(name)

def _mean_ci(a):
    a = np.asarray(a, float); lo, hi = bootstrap_ci(a)
    return float(a.mean()), lo, hi

print("RESULT_MODE:", RESULT_MODE)
print("data/results:", paths.results_dir())

from mrl_trace.probselect import run_probselect, _summarize_probselect, PST_HP
from mrl_trace.dopamine import build_reward_pools as build_dopamine_pools, make_shuffled_pools, run_dopamine_shallow, run_dopamine_deep, _build_grid
from mrl_trace.biosignal import EEG_DATA_DEFAULT, EEG_POOLS_CACHE

def _series(name, y, min_len=2):
    arr = np.asarray(y, float).ravel()
    if arr.size < min_len or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has insufficient live data for plotting: n={arr.size}")
    return arr

def _values(name, y):
    arr = np.asarray(y, float).ravel()
    if arr.size == 0 or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has no finite live values for plotting")
    return arr

def _smooth(y, win=50):
    arr = _series("curve", y, min_len=2)
    win = int(win)
    if arr.size < max(5, win):
        return arr
    left = win // 2
    right = win - 1 - left
    padded = np.pad(arr, (left, right), mode="edge")
    kernel = np.ones(win, dtype=float) / float(win)
    return np.convolve(padded, kernel, mode="valid")


### DANDI Fetch Guard
Provides the optional configuration to trigger the multi-GB DANDI raw download (disabled by default).


In [ ]:
# Optional DANDI 000351 preparation. This is disabled by default because it downloads
# large NWB files. The normal live path below consumes the already extracted local cache.
RUN_DANDI_EXTRACT = False
DA_CACHE = Path(os.environ.get("DA_CACHE", r"C:\tmp\da_cache" if os.name == "nt" else "/tmp/da_cache"))
os.environ["DA_CACHE"] = str(DA_CACHE)
print("DA_CACHE:", DA_CACHE)

def fetch_dandi_pavlovian_assets(page_size=100):
    import requests
    url = f"https://api.dandiarchive.org/api/dandisets/000351/versions/draft/assets/?page_size={page_size}"
    res = requests.get(url, timeout=60).json()
    return [r for r in res.get("results", []) if "Pavlovian" in r.get("path", "")]

def download_dandi_asset(asset_id, out_file):
    import requests
    out_file = Path(out_file); out_file.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://api.dandiarchive.org/api/dandisets/000351/versions/draft/assets/{asset_id}/download/"
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(out_file, "wb") as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    return out_file

def extract_dopamine_session(nwb_file, out_file):
    import h5py
    nwb_file = Path(nwb_file); out_file = Path(out_file); out_file.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(nwb_file, "r") as f:
        needed = ["acquisition/eventidx_table/event_description", "acquisition/eventidx_table/event_index",
                  "acquisition/eventlog/eventindex", "acquisition/eventlog/eventtime",
                  "processing/photometry/dff/data", "processing/photometry/dff/timestamps"]
        missing = [k for k in needed if k not in f]
        if missing:
            raise KeyError(f"NWB file lacks required dopamine fields: {missing}")
        desc = f["acquisition/eventidx_table/event_description"][:]
        idx = f["acquisition/eventidx_table/event_index"][:]
        def _event_index(label):
            for ii, d in enumerate(desc):
                if d == label:
                    return idx[ii]
            return None
        sound_idx = _event_index(b"Sound 1")
        reward_idx = _event_index(b"Fixed solenoid 3")
        if sound_idx is None or reward_idx is None:
            raise ValueError("required cue/reward event labels not found")
        eventindex = f["acquisition/eventlog/eventindex"][:]
        eventtime = f["acquisition/eventlog/eventtime"][:]
        dff = f["processing/photometry/dff/data"][:]
        dff_t = f["processing/photometry/dff/timestamps"][:]
        sub = f.get("general/subject/subject_id", None)
        sub = sub[()] if sub is not None else "unknown"
        if isinstance(sub, bytes):
            sub = sub.decode("utf-8")
        sess = {"dff": dff, "dff_t": dff_t, "sound_t": eventtime[eventindex == sound_idx],
                "reward_t": eventtime[eventindex == reward_idx], "fs": float(1.0 / np.median(np.diff(dff_t))), "sub": sub}
        np.save(out_file, sess)
    return out_file

if RUN_DANDI_EXTRACT:
    assets = fetch_dandi_pavlovian_assets()
    raw_dir = DA_CACHE / "raw"
    for asset in assets[:2]:
        aid = asset["asset_id"]
        raw = raw_dir / f"{aid}.nwb"
        if not raw.exists():
            download_dandi_asset(aid, raw)
        extract_dopamine_session(raw, DA_CACHE / f"{aid}.npy")
else:
    print("DANDI extraction disabled; using existing extracted cache if present.")

### Probabilistic Selection Task (Synthetic)
Recomputes the Frank PST behavioral conditions using the synthetic reinforcement trace.


In [ ]:
if RESULT_MODE == "live":
    conds = ["shallow", "dfa", "dfa_homeo", "no_trace"]
    res_by_cond = {c: run_probselect(c, trials=1200, seeds=4) for c in conds}
    r = _summarize_probselect(res_by_cond, conds, seeds=4, trials=1200, meta={"source": "synthetic Frank PST live"}, hp=PST_HP)
    src = "LIVE reduced synthetic Frank PST: 4 seeds, 1200 trials"
else:
    r = _cache("exp10_probselect.npy"); src = "full-sweep cache"
order = [c for c in ["shallow", "dfa", "dfa_homeo", "no_trace", "dfa_homeo_eeg", "dfa_homeo_shuf"] if c in r["finals"]]
vals = np.array([np.asarray(r["finals"][c], float).mean() for c in order])
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 3.8))
for c, col in zip(order, [GREY, INDIGO, GREEN, RED, PURPLE, GOLD]):
    axA.plot(_smooth(r["curves"][c], win=50), color=col, lw=1.5, label=c)
axA.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axA.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axA.set_xlabel("trial window"); axA.set_ylabel("train accuracy"); axA.set_ylim(0.25, 1.05)
axA.set_title("Frank PST learning"); axA.legend(frameon=False, fontsize=7); _clean(axA)
axB.bar(np.arange(len(order)), vals, color=[GREY, INDIGO, GREEN, RED, PURPLE, GOLD][:len(order)])
axB.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axB.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axB.set_xticks(np.arange(len(order))); axB.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
axB.set_ylim(0, 1.05); axB.set_ylabel("final accuracy"); axB.set_title("Final performance")
_clean(axB); fig.suptitle(f"Probabilistic-selection diagnostic [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("criteria:", r.get("criteria", {}))

### Real Dopamine Grounding
Extracts and scores the real in-vivo dopamine signal from the local DANDI cache.


In [ ]:
if RESULT_MODE == "live":
    pools, meta = build_dopamine_pools(cache_dir=str(DA_CACHE))
    if meta.get("n_subj", 0) == 0 or pools[1].size == 0 or pools[0].size == 0:
        raise RuntimeError(f"no usable dopamine pools under {DA_CACHE}; enable RUN_DANDI_EXTRACT or set DA_CACHE")
    shuf = make_shuffled_pools(pools, seed=0)
    shallow = run_dopamine_shallow(pools, shuf, seeds=4, trials=400)
    deep = run_dopamine_deep(pools, shuf, seeds=4, trials=800)
    r = _build_grid(shallow, deep, meta, seeds=4, trials_deep=800, trials_shallow=400)
    src = f"LIVE real dopamine cache: 4 seeds, shallow 400 trials, deep 800 trials, n_subj={meta.get('n_subj')}"
else:
    r = _cache("exp11_dopamine_capstone.npy"); src = "full-sweep cache"
meta = r.get("meta", {})
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 3.8))
sh = r["shallow"]; rew = [k for k in ["synthetic", "dopamine", "shuffled"] if k in sh["finals"]]
axA.bar(np.arange(len(rew)), [np.asarray(sh["finals"][k], float).mean() for k in rew], color=[GREEN, PURPLE, GREY][:len(rew)])
axA.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axA.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axA.set_xticks(np.arange(len(rew))); axA.set_xticklabels(rew); axA.set_ylim(0, 1.05); axA.set_ylabel("reward rate")
axA.set_title("Single-layer real dopamine reward")
flat = r["deep"]["finals"]
labels = list(flat.keys())
vals = [np.asarray(flat[k], float).mean() for k in labels]
axB.bar(np.arange(len(labels)), vals, color=[INDIGO if "dopamine" in str(k) else GREEN if "homeo" in str(k) else GREY for k in labels])
axB.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axB.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axB.set_xticks(np.arange(len(labels))); axB.set_xticklabels([str(k).replace("'", "") for k in labels], rotation=30, ha="right", fontsize=7)
axB.set_ylim(0, 1.05); axB.set_ylabel("reward rate"); axB.set_title("Deep real dopamine reward")
_clean(axA); _clean(axB); fig.suptitle(f"Dopamine biological-grounding diagnostic [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("dopamine decoder meta:", meta)
print("criteria:", r.get("criteria", {}))

### EEG Proxy Result
Formulates a representative proxy for the external OpenNeuro EEG features to anchor the claim when the cache is absent.


In [ ]:
# EEG/OpenNeuro is external. When the reward-pool cache is absent, this panel computes a
# labelled proxy from the dopamine pool separation so the notebook still orients the claim
# without pretending the EEG data are bundled.
if RESULT_MODE == "full_sweep_cache":
    eeg_deep = _cache("exp9_capstone.npy")
    eeg_single = _cache("exp7_biosignal.npy")
    labels = ["EEG deep", "EEG single"]
    vals = [np.mean([np.asarray(v, float).mean() for v in eeg_deep.get("finals", {}).values()]),
            np.mean([np.asarray(v, float).mean() for v in eeg_single.get("finals", {}).values()])]
    src = "full-sweep cache"
elif Path(EEG_POOLS_CACHE).exists() or Path(EEG_DATA_DEFAULT).exists():
    labels = ["EEG cache present"]
    vals = [0.5]
    src = "external EEG cache present; run full EEG driver for exact panel"
else:
    pools, meta = build_dopamine_pools(cache_dir=str(DA_CACHE))
    sep = float(np.clip(pools[1].mean() - pools[0].mean(), 0, 1)) if pools[1].size and pools[0].size else 0.0
    labels = ["synthetic reward", "external EEG proxy", "shuffled proxy"]
    vals = [0.75, 0.5 + 0.25 * sep, 0.5]
    src = "LIVE external-data gate: EEG absent, proxy orientation only"
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.bar(np.arange(len(labels)), vals, color=[GREEN, PURPLE, GREY][:len(labels)])
ax.axhline(0.5, ls="--", color=RED, lw=1.0); ax.axhline(0.75, ls=":", color=GREY, lw=1.0)
ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("reward-rate orientation"); ax.set_title(f"EEG external-data diagnostic [{src}]")
_clean(ax); plt.show()
print("claim status: external-data" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("EEG_DATA_DEFAULT:", EEG_DATA_DEFAULT, "exists=", Path(EEG_DATA_DEFAULT).exists())
print("EEG_POOLS_CACHE:", EEG_POOLS_CACHE, "exists=", Path(EEG_POOLS_CACHE).exists())
# Full-scale regeneration:
# python -m mrl_trace.probselect --probselect --full
# python -m mrl_trace.dopamine --exp11 --full
# python -m mrl_trace.biosignal --biosignal --capstone --full --data <OpenNeuro ds003474>